In [1]:
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

print("Libraries loaded successfully!")

/Users/ayberkpalta/Desktop/llm_end/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded successfully!


In [2]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)

print("MPNet embedding model hazır!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11821.13it/s]


MPNet embedding model hazır!


MP-NET

In [3]:
test_text = "This paper proposes a deep learning method for image classification."

test_embedding = embedding_model.encode(test_text)

print("Embedding shape:", test_embedding.shape)
print("İlk 10 değer:", test_embedding[:10])

Embedding shape: (768,)
İlk 10 değer: [-3.1417679e-02  2.9722229e-02 -1.9013915e-02  9.8026834e-02
 -3.7779368e-02  1.9264134e-02  8.4034592e-02  1.1420936e-02
 -5.6752655e-05 -1.1908741e-02]


In [4]:
import json
import os

# Proje ana dizinine göre yollar
train_path = "../data/processed/train_papers.json"
validation_path = "../data/processed/validation_papers.json"
test_path = "../data/processed/test_papers.json"

# Train
with open(train_path, "r", encoding="utf-8") as f:
    train_papers = json.load(f)

# Validation
with open(validation_path, "r", encoding="utf-8") as f:
    validation_papers = json.load(f)

# Test
with open(test_path, "r", encoding="utf-8") as f:
    test_papers = json.load(f)

print("Train:", len(train_papers))
print("Validation:", len(validation_papers))
print("Test:", len(test_papers))
print("Total:", len(train_papers) + len(validation_papers) + len(test_papers))

Train: 124
Validation: 26
Test: 29
Total: 179


In [5]:
print("TRAIN PAPER:")
print(train_papers[0])

print("\nVALIDATION PAPER:")
print(validation_papers[0])

print("\nTEST PAPER:")
print(test_papers[0])

TRAIN PAPER:
{'id': 'http://arxiv.org/abs/2012.10055v2', 'title': 'End-to-End Speaker Diarization as Post-Processing', 'abstract': "This paper investigates the utilization of an end-to-end diarization model as post-processing of conventional clustering-based diarization. Clustering-based diarization methods partition frames into clusters of the number of speakers; thus, they typically cannot handle overlapping speech because each frame is assigned to one speaker. On the other hand, some end-to-end diarization methods can handle overlapping speech by treating the problem as multi-label classification. Although some methods can treat a flexible number of speakers, they do not perform well when the number of speakers is large. To compensate for each other's weakness, we propose to use a two-speaker end-to-end diarization method as post-processing of the results obtained by a clustering-based method. We iteratively select two speakers from the results and update the results of the two spea

In [6]:
def create_texts(papers):
    texts = []
    labels = []

    for paper in papers:
        title = paper.get("title", "")
        abstract = paper.get("abstract", "")
        
        # Title + Abstract
        text = f"{title}. {abstract}"
        
        texts.append(text)
        labels.append(paper["labels"])
    
    return texts, labels


# Train
train_texts, train_labels = create_texts(train_papers)

# Validation
validation_texts, validation_labels = create_texts(validation_papers)

# Test
test_texts, test_labels = create_texts(test_papers)


print("Train texts:", len(train_texts))
print("Validation texts:", len(validation_texts))
print("Test texts:", len(test_texts))

print("\nExample text:")
print(train_texts[0][:1000])

print("\nExample labels:")
print(train_labels[0])

Train texts: 124
Validation texts: 26
Test texts: 29

Example text:
End-to-End Speaker Diarization as Post-Processing. This paper investigates the utilization of an end-to-end diarization model as post-processing of conventional clustering-based diarization. Clustering-based diarization methods partition frames into clusters of the number of speakers; thus, they typically cannot handle overlapping speech because each frame is assigned to one speaker. On the other hand, some end-to-end diarization methods can handle overlapping speech by treating the problem as multi-label classification. Although some methods can treat a flexible number of speakers, they do not perform well when the number of speakers is large. To compensate for each other's weakness, we propose to use a two-speaker end-to-end diarization method as post-processing of the results obtained by a clustering-based method. We iteratively select two speakers from the results and update the results of the two speakers to impro

mpnet embed

In [7]:
train_embeddings = embedding_model.encode(
    train_texts,
    show_progress_bar=True
)

validation_embeddings = embedding_model.encode(
    validation_texts,
    show_progress_bar=True
)

test_embeddings = embedding_model.encode(
    test_texts,
    show_progress_bar=True
)

print("Train embedding shape:", train_embeddings.shape)
print("Validation embedding shape:", validation_embeddings.shape)
print("Test embedding shape:", test_embeddings.shape)

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.28s/it]

Train embedding shape: (124, 768)
Validation embedding shape: (26, 768)
Test embedding shape: (29, 768)


In [8]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(
    classes=[
        "NLP",
        "Computer Vision",
        "Machine Learning",
        "Robotics"
    ]
)

Y_train = mlb.fit_transform(train_labels)
Y_validation = mlb.transform(validation_labels)
Y_test = mlb.transform(test_labels)

print("Classes:", mlb.classes_)

print("Y_train shape:", Y_train.shape)
print("Y_validation shape:", Y_validation.shape)
print("Y_test shape:", Y_test.shape)

print("\nFirst 5 train labels:")
print(Y_train[:5])

Classes: ['NLP' 'Computer Vision' 'Machine Learning' 'Robotics']
Y_train shape: (124, 4)
Y_validation shape: (26, 4)
Y_test shape: (29, 4)

First 5 train labels:
[[1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 1 0]]


In [9]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

embedding_classifier = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=2000
    )
)

embedding_classifier.fit(
    train_embeddings,
    Y_train
)

print("MPNet + Logistic Regression classifier trained successfully!")

MPNet + Logistic Regression classifier trained successfully!


In [10]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

embedding_classifier = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        max_iter=2000
    )
)

embedding_classifier.fit(
    train_embeddings,
    Y_train
)

print("Embedding classifier trained successfully!")

Embedding classifier trained successfully!


Validation tahminleri.

In [11]:
validation_probabilities = embedding_classifier.predict_proba(
    validation_embeddings
)

print("Probability shape:", validation_probabilities.shape)

print("\nFirst 5 validation probabilities:")
print(validation_probabilities[:5])

Probability shape: (26, 4)

First 5 validation probabilities:
[[0.6434975  0.14413828 0.24585961 0.105219  ]
 [0.34475592 0.14376202 0.2916665  0.16946256]
 [0.63435435 0.10872481 0.21994634 0.11673663]
 [0.71526676 0.10845139 0.19865696 0.11184746]
 [0.68537134 0.13682112 0.2462035  0.11323205]]


Choose Threshold

In [12]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]

print("=" * 70)
print("MPNET — VALIDATION THRESHOLD ANALYSIS")
print("=" * 70)

for threshold in thresholds:

    validation_predictions = (
        validation_probabilities >= threshold
    ).astype(int)

    micro_f1 = f1_score(
        Y_validation,
        validation_predictions,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        Y_validation,
        validation_predictions,
        average="macro",
        zero_division=0
    )

    predicted_positives = validation_predictions.sum()

    print(
        f"Threshold: {threshold:.2f} | "
        f"Micro F1: {micro_f1:.3f} | "
        f"Macro F1: {macro_f1:.3f} | "
        f"Predicted Positives: {predicted_positives}"
    )

MPNET — VALIDATION THRESHOLD ANALYSIS
Threshold: 0.20 | Micro F1: 0.659 | Macro F1: 0.674 | Predicted Positives: 54
Threshold: 0.25 | Micro F1: 0.754 | Macro F1: 0.758 | Predicted Positives: 41
Threshold: 0.30 | Micro F1: 0.842 | Macro F1: 0.837 | Predicted Positives: 29
Threshold: 0.35 | Micro F1: 0.852 | Macro F1: 0.846 | Predicted Positives: 26
Threshold: 0.40 | Micro F1: 0.800 | Macro F1: 0.782 | Predicted Positives: 22
Threshold: 0.45 | Micro F1: 0.696 | Macro F1: 0.643 | Predicted Positives: 18
Threshold: 0.50 | Micro F1: 0.696 | Macro F1: 0.643 | Predicted Positives: 18


test

In [13]:
test_probabilities = embedding_classifier.predict_proba(
    test_embeddings
)

print("Test probability shape:", test_probabilities.shape)

print("\nFirst 5 test probabilities:")
print(test_probabilities[:5])

Test probability shape: (29, 4)

First 5 test probabilities:
[[0.34056324 0.14131063 0.35733715 0.20570867]
 [0.5755085  0.12525353 0.29933265 0.11666794]
 [0.717439   0.10154631 0.21434891 0.10537683]
 [0.71059084 0.12465256 0.2143079  0.09721033]
 [0.59685636 0.1116588  0.34294972 0.10780704]]


final test pred

In [14]:
TEST_THRESHOLD = 0.35

test_predictions = (
    test_probabilities >= TEST_THRESHOLD
).astype(int)

print("=" * 70)
print("FINAL MPNET TEST PREDICTIONS")
print("=" * 70)

print("Threshold:", TEST_THRESHOLD)
print("Predicted positives:", test_predictions.sum())

print("\nFirst 5 predictions:")
print(test_predictions[:5])

FINAL MPNET TEST PREDICTIONS
Threshold: 0.35
Predicted positives: 32

First 5 predictions:
[[0 0 1 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]
 [1 0 0 0]]


In [15]:
from sklearn.metrics import (
    f1_score,
    classification_report
)

micro_f1 = f1_score(
    Y_test,
    test_predictions,
    average="micro",
    zero_division=0
)

macro_f1 = f1_score(
    Y_test,
    test_predictions,
    average="macro",
    zero_division=0
)

print("=" * 70)
print("FINAL MPNET TEST RESULTS")
print("=" * 70)

print(f"Threshold: {TEST_THRESHOLD}")
print(f"Micro F1: {micro_f1:.3f}")
print(f"Macro F1: {macro_f1:.3f}")

print("\nClassification Report:")

print(
    classification_report(
        Y_test,
        test_predictions,
        target_names=mlb.classes_,
        zero_division=0
    )
)

FINAL MPNET TEST RESULTS
Threshold: 0.35
Micro F1: 0.812
Macro F1: 0.812

Classification Report:
                  precision    recall  f1-score   support

             NLP       1.00      0.88      0.93         8
 Computer Vision       0.78      0.88      0.82         8
Machine Learning       0.71      0.62      0.67         8
        Robotics       0.78      0.88      0.82         8

       micro avg       0.81      0.81      0.81        32
       macro avg       0.82      0.81      0.81        32
    weighted avg       0.82      0.81      0.81        32
     samples avg       0.83      0.83      0.82        32

